In [ ]:
import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

import os
from pathlib import Path
from pyspark.sql import functions as F

In [ ]:
%%time
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=56,
    executor_memory="32g",
    executor_cores=4,
    # update_configs={
    #     "spark.sql.shuffle.partitions": "512",  # 6.2B ÷ 500 = 12.4M rows/partition
    # },
    aws_profile="admin-user"
)

In [ ]:
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

In [ ]:
base_dir = "/data/data_processing/backfill-nwm/nwm-forcing-teehr-warehouse/local/cache/fetching"

## Primary
# timeseries_type = "primary"  # just for reference (unused here otherwise)
# config_name = "nwm30_forcing_analysis_assim"
# config_name = "nwm30_forcing_analysis_assim_alaska"
# config_name = "nwm30_forcing_analysis_assim_extend"
# config_name = "nwm30_forcing_analysis_assim_extend_alaska"
# config_name = "nwm30_forcing_analysis_assim_hawaii"
# config_name = "nwm30_forcing_analysis_assim_puertorico"

## Secondary
timeseries_type = "secondary"
# config_name = "nwm30_forcing_medium_range"
# config_name = "nwm30_forcing_medium_range_alaska"
# config_name = "nwm30_forcing_short_range_alaska"
# config_name = "nwm30_forcing_short_range_hawaii"
config_name = "nwm30_forcing_short_range_puertorico"

cache_dir = Path(base_dir, config_name)
cache_dir

In [ ]:
files = list(Path(cache_dir).glob("**/*.parquet"))
total_size = sum(f.stat().st_size for f in files)
print(f"Files: {len(files)}")
print(f"Total size: {total_size / (1024**3):.2f} GB")

In [ ]:
nwm_sdf = spark.read.option("recursiveFileLookup", "true").parquet(cache_dir.as_posix())
nwm_sdf.count()

In [ ]:
nwm_sdf.select(F.min("value_time"), F.max("value_time")).show()

In [ ]:
nwm_sdf = nwm_sdf.withColumn("variable_name", F.lit("rainrate_hourly_mean"))
nwm_sdf.show(n=5, truncate=False)

In [ ]:
timeseries_type

In [ ]:
%%time
ev.secondary_timeseries.load_dataframe(nwm_sdf)